# YOLO + OCR Plate Detection - Colab Notebook

- install the dependencies
- load the YOLO model
- detect the plates
- read text using EasyOCR
- process a single image and a full folder
- save the results in an `outputs` folder

In [ ]:
# Cell 1 - Install dependencies
# Run this cell when starting the notebook.

!pip install -q ultralytics easyocr opencv-python-headless numpy pandas

print('Dependencies installed.')

In [ ]:
# Cell 2 - Check the repository/project
# If the repo is not present, clone it from GitHub.

import os
from pathlib import Path

REPO_URL = "https://github.com/senisama/AI-UAV-Address-Verification.git"
WORKDIR = Path("/content")
PROJECT_DIR = WORKDIR / "AI-UAV-Address-Verification"

if not PROJECT_DIR.exists():
    %cd /content
    !git clone {REPO_URL}

%cd {PROJECT_DIR}

print(f"Project available at: {PROJECT_DIR}")
print("Directory tree:")
!ls -la

In [ ]:
# Cell 3 - Define the working paths
# Upload a local image in the next cell, then use that path here.

from pathlib import Path

PROJECT_ROOT = Path('/content/AI-UAV-Address-Verification')
MODEL_PATH = PROJECT_ROOT / 'models' / 'best.pt'
IMAGE_PATH = PROJECT_ROOT / 'uploaded_image.jpg'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('MODEL_PATH =', MODEL_PATH)
print('IMAGE_PATH =', IMAGE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

In [ ]:
# Cell 4 - Import the pipeline

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from plate_ocr.detect_and_ocr import PlateOCRPipeline

print('Pipeline imported successfully.')

In [ ]:
# Cell 5 - Initialize the pipeline

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

pipeline = PlateOCRPipeline(
    model_path=MODEL_PATH,
    languages=('en',),
    device=DEVICE,
)

print('YOLO + EasyOCR are ready.')

In [ ]:
# Cell 6 - Test on a single uploaded image
# First upload your image using the next cell, then this cell will process it.

from google.colab import files
from pathlib import Path

uploaded = files.upload()
if uploaded:
    uploaded_file_name = next(iter(uploaded))
    uploaded_path = Path('/content') / uploaded_file_name
    IMAGE_PATH.write_bytes(uploaded[uploaded_file_name])
    print('Uploaded image saved to:', IMAGE_PATH)
else:
    print('No file selected.')

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"No uploaded image found at {IMAGE_PATH}")

print('Processing:', IMAGE_PATH)
detections, output_path = pipeline.detect_and_read(IMAGE_PATH, conf_threshold=0.25)

print(f"Number of detections: {len(detections)}")
print('Annotated image saved to:', output_path)


for i, det in enumerate(detections, start=1):display(Image(filename=str(output_path), width=800))

    print(f"Plate {i}: bbox={det['bbox']} confidence={det['confidence']:.3f} texts={det['texts']}")

from IPython.display import Image, display

In [ ]:
# Cell 7 - Process a full folder

from pathlib import Path

results = pipeline.detect_and_read_directory(RAW_DIR / 'images', conf_threshold=0.25)

print(f"Images processed: {len(results)}")
for item in results:
    print(item['image'], '->', len(item['detections']), 'detection(s)')
    for det in item['detections']:
        print('  ', det['bbox'], det['texts'])

In [ ]:
# Cell 8 - Export results to a table

import pandas as pd

rows = []
for item in results:
    for det in item['detections']:
        rows.append({
            'image': item['image'],
            'bbox': det['bbox'],
            'confidence': det['confidence'],
            'texts': ' | '.join(det.get('texts', [])),
            'output': str(item['output'])
        })

if rows:
    df = pd.DataFrame(rows)
    print(df.head())
    summary_path = PROJECT_ROOT / 'outputs' / 'results_summary.csv'
    df.to_csv(summary_path, index=False)
    print('CSV saved to:', summary_path)
else:
    print('No detections found.')